In [17]:
import os
import csv

os.chdir('/Users/rv/Projects/7CS074') # Change to the project root directory

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_score
from datetime import datetime

from sklearn.preprocessing import OneHotEncoder
from category_encoders import TargetEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [18]:
BASE_DIR = os.getcwd()

DATA_RAW_PATH = os.path.join(BASE_DIR, "data/raw")
DATA_CLEAN_PATH = os.path.join(BASE_DIR, "data/clean")

DATASET_CLEAN_FILE_PATH = os.path.join(DATA_CLEAN_PATH, "cleaned_dataset.csv")

COLUMNS = [
    "make",
    "model",
    "year",
    "price",
    "transmission",
    "mileage",
    "fuelType",
    "tax",
    "mpg",
    "engineSize",
]

NUMERIC_OUTLIER_COLUMNS = [
    "price",
    "mileage",
    "mpg",
    "engineSize",
    "tax"
]

LOW_CATEGORICAL_FEATURES_OVERALL = [
    'transmission',
    'fuelType'
]

HIGH_CATEGORICAL_FEATURES_GLOBAL = [
    'make',
    'model'
]

HIGH_CATEGORICAL_FEATURES_PER_MAKE = [
    'model'
]

NUMERIC_FEATURES = [
    'year', 
    'tax', 
    'mileage',
    'engineSize',
    'mpg',
    
    # We are making this based on the (mpg/engine size)
    'efficiency_score',
    # We are making this base on (current_year - year)
    'vehicle_age'
]

EXTRA_COLUMNS = ['mileage2', 'fuel type2', 'engine size2', 'reference']
EXPECTED_COLUMNS = 9

In [19]:
def classification_metrics(y_true, y_pred, average='weighted') -> dict:
    """Calculate classification metrics"""
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average=average, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred)
    }

def get_classification_report(y_true, y_pred, target_names=None):
    """Get detailed classification report"""
    return classification_report(
        y_true, 
        y_pred, 
        target_names=target_names,
        zero_division=0
    )

def select_best_model_cv(
    X,
    y,
    models: dict,
    cv_splits: int = 5,
    scoring: str = "neg_mean_absolute_error"
):
    """
        Selects the best model using cross-validation.
    """
    cv = KFold(n_splits=cv_splits, shuffle=True, random_state=42)

    scores = {}
    for name, model in models.items():
        cv_score = cross_val_score(
            clone(model),
            X,
            y,
            cv=cv,
            scoring=scoring,
            n_jobs=-1
        ).mean()
        scores[name] = cv_score

    best_model_name = max(scores, key=scores.get)
    best_model = clone(models[best_model_name])
    best_model.fit(X, y)

    return best_model_name, best_model, scores

In [20]:
def engineer_features(
    df: pd.DataFrame,
    y: pd.Series,
    low_card_categorical_features,
    high_card_categorical_features,
):
    """
        Returns a joint dataframe and the encoder of both numerical values and categorical after one hot encoding and target encoding values like can be see in src/global_vars/ - HIGH_CARD_CATEGORICAL_FEATURES, LOW_CARD_CATEGORICAL_FEATURES.
        Creates efficiency score to remove the stress on training on two features (mpg and engine size) to determine price, instead just create efficiency score to let model decide on that.
        Creates vehicle age to better determine the age of vehicle instead of year of manufacture.
    """
    df = df.copy()
    
    # Efficiency score, better for the model, instead of trying to race over which feature decides on the price, just create the efficiency score to find that out per make / model of vehicle
    # Will onl create if engineSize is higher than 0, making sure we do not create bad data for model to be trained, so we just set as nan if logic fails
    if {'mpg', 'engineSize'}.issubset(df.columns):
        df['efficiency_score'] = np.where(
            df['engineSize'] > 0,
            df['mpg'] / df['engineSize'],
            np.nan
        )
    
    # We will create vehicle_age as the primary column to create a better relationship between price and age of vehicle
    if {'year'}.issubset(df.columns):
        current_year = datetime.now().year
        df['vehicle_age'] = np.where(
            df['year'] > 0,
            current_year - df['year'],
            np.nan 
        )

    X_numeric = df[NUMERIC_FEATURES]

    low_card_encoder = OneHotEncoder(
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    )
    low_card_encoder.fit(df[low_card_categorical_features])
    X_train_low_card = low_card_encoder.transform(df[low_card_categorical_features])
    low_categorical_cols = low_card_encoder.get_feature_names_out(
        low_card_categorical_features
    )
    X_low_df = pd.DataFrame(
        X_train_low_card,
        columns=low_categorical_cols,
        index=df.index
    )
    
    
    high_card_encoder = TargetEncoder(
        cols=high_card_categorical_features,
        smoothing=10
    )
    X_train_high_card = high_card_encoder.fit_transform(df[high_card_categorical_features], y)
    high_categorical_cols = high_card_encoder.get_feature_names_out(
        high_card_categorical_features
    )
    X_high_df = pd.DataFrame(
        X_train_high_card,
        columns=high_categorical_cols,
        index=df.index
    )
    
    X_Concatenated = pd.concat(
        [
            X_numeric, 
            X_low_df,
            X_high_df
        ],
        axis=1
    )
    
    X_Concatenated.drop(['year', 'mpg', 'engineSize'], axis=1, inplace=True)

    # Returns a joint dataframe of both numeric X-Axis and categorical X-Axis
    return X_Concatenated

def get_feature_effects(model):
    """
        Returns (values, kind) for feature effect visualization. Since we are using different models, we need to extract feature importance or coefficients based on model type.
        Supported models:
        - Tree models -> feature_importances_ - https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html
        - Linear models -> abs(coef_)
    """
    # RandomForest, GradientBoosting, etc.
    if hasattr(model, "feature_importances_"):
        values = model.feature_importances_
        return values, "importance"

    # LinearRegression, Ridge, Lasso, etc.
    if hasattr(model, "coef_"):
        values = np.abs(model.coef_)
        return values, "coefficient"

    # Unsupported model
    return None, None

In [21]:
def get_classification_candidate_models(random_state=42):
    """
        Returns a list of different classification models, starting with a base model, and hyper tunning others, this can allow to find which is the best model in the case of classification for our current dataset of used cars
    """
    return {
        "random_forrest": RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=random_state
        ),
        "decision_tree": DecisionTreeClassifier(random_state=random_state, class_weight="balanced"),
    }


In [22]:
# First we will check and run the preprocessing script
# In case this is already done, this will not overwrite existing files
def process_raw_multiple_data_files():
    if not os.path.exists(DATA_RAW_PATH):
        raise FileNotFoundError("Data directory not found. Please ensure the project structure is correct.")

    # Creates if not exists
    if not os.path.exists(DATA_CLEAN_PATH):
        os.makedirs(DATA_CLEAN_PATH)

    if os.path.exists(DATASET_CLEAN_FILE_PATH):
        return  # If cleaned dataset already exists, skip processing

    # Creates if not exists, double check at this point
    if not os.path.exists(DATASET_CLEAN_FILE_PATH):
        os.makedirs(DATA_CLEAN_PATH, exist_ok=True)

    directory_raw_bytes = os.fsencode(DATA_RAW_PATH)

    # Validate that CSV files to see if exists and is non-empty
    for file in os.listdir(directory_raw_bytes):
        filename = os.fsdecode(file)
        if filename.endswith(".csv"):
            DATASET_PATH = os.path.join(DATA_RAW_PATH, filename)
            df = pd.read_csv(DATASET_PATH)
            if df.empty:
                raise ValueError("Loaded dataset is empty. Please check the dataset file.")
            continue
        else:
            continue

    # Concatenate all CSV files in the raw data directory
    # We will read the clean path / file directly, and add the default headers
    # We will use our COLUMNS variable as the schema
    schema = COLUMNS
    with open(DATASET_CLEAN_FILE_PATH, 'w') as csvfile:
        writer = csv.writer(csvfile, delimiter=',')
        writer.writerow([g for g in schema])

    for file in os.listdir(directory_raw_bytes):
        filename = os.fsdecode(file)
        if filename.endswith(".csv"):
            DATASET_PATH = os.path.join(DATA_RAW_PATH, filename)
            df_default = pd.read_csv(DATASET_PATH)
            df = clean_data(df_default)

            if not df.empty:
                make = filename.replace('.csv', '')
                df.insert(0, 'make', make) # Insert 'make' as first column
                
                # Only keep rows with exactly EXPECTED_COLUMNS + 1 columns after inserting 'make'
                df = df[df.apply(lambda x: len(x) == EXPECTED_COLUMNS + 1, axis=1)]

                df.to_csv(
                    DATASET_CLEAN_FILE_PATH,
                    mode='a',
                    header=False,
                    index=False
                )
        else:
            continue

def remove_iqr_outliers(
    df: pd.DataFrame,
    columns: list,
    factor: float = 1.5
) -> pd.DataFrame:
    """
        Remove outliers using Interquartile Range (IQR) method for selected columns.
    """
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            continue

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR

        df = df[(df[col] >= lower) & (df[col] <= upper)]

    return df

def apply_domain_constraints(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
        Apply domain constraints to the dataframe, meaning that we filter the data based on known valid ranges for each numeric column, so that we remove any rows that have values outside these ranges.
    """
    df = df.copy()

    constraints = {
        'price': lambda x: (x > 100) & (x < 200_000),
        'mileage': lambda x: (x >= 0) & (x < 300_000),
        'engineSize': lambda x: (x > 0) & (x < 12.0),
        'mpg': lambda x: (x > 5) & (x < 60),
        'tax': lambda x: (x >= 0)
    }

    for col, condition in constraints.items():
        if col in df.columns:
            df = df[condition(df[col])]

    return df

def coerce_numeric_columns(
    df: pd.DataFrame, 
    columns: list
) -> pd.DataFrame:
    """
        Here we make sure that any columns parse we parse to integers / numeric values, if record in column is string of currency numeric, we remove this regex
    """
    df = df.copy()

    for col in columns:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(r"[£,]", "", regex=True)
                .str.strip()
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

# Cleans the data, and returns copy of cleaned dataframe
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
        Here we clean out the data from any possible problems which can cause bad training
    """
    df = df.copy()
    
    # Remove extra columns and duplicate columns if present
    df.drop(columns=[col for col in EXTRA_COLUMNS if col in df.columns], inplace=True, errors='ignore')
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Only keep rows with exactly EXPECTED_COLUMNS columns
    df = df[df.apply(lambda x: len(x) == EXPECTED_COLUMNS, axis=1)]
    
    # Replace 'N/A' and Drop record duplicates
    # If by any chance 'N/A' is a string by text, we shall change it to nan 'na', later we will remove anyway
    df.replace("N/A", np.nan, inplace=True)
    df.drop_duplicates(inplace=True)

    # Drop rows that are all empty or just commas 
    # """
    #     ,,,,,,,,,,
    # """
    df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    df.dropna(how='all', inplace=True)

    # Coerce numeric columns
    df = coerce_numeric_columns(df, NUMERIC_OUTLIER_COLUMNS)
    # Domain filtering
    df = apply_domain_constraints(df)
    # Statistical outliers
    df = remove_iqr_outliers(
        df,
        columns=[c for c in NUMERIC_OUTLIER_COLUMNS if c in df.columns]
    )
    
    return df

process_raw_multiple_data_files()

In [23]:
import numpy as np

if not os.path.exists(DATASET_CLEAN_FILE_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATASET_CLEAN_FILE_PATH}. Please ensure the dataset is placed correctly.")

df = pd.read_csv(DATASET_CLEAN_FILE_PATH, sep=',', engine='python') # read with proper delimiter handling, and with python engine always
if df.empty:
    raise ValueError("Loaded dataset is empty. Please check the dataset file.")

print(f"Data loaded successfully from {DATASET_CLEAN_FILE_PATH}.")

target_col_class = 'price_category'

min_samples_per_make=300
cv_splits=5

trained_models_per_make = {}

df_copy = df.copy()
df_copy['price_category'] = pd.cut(
    df_copy['price'],
    bins=[0, 10000, 20000, 30000, np.inf],
    labels=['Budget', 'Mid-Range', 'Premium', 'Luxury']
)

# Remove rows with NaN price_category
df_copy = df_copy.dropna(subset=['price_category'])

Data loaded successfully from /Users/rv/Projects/7CS074/data/clean/cleaned_dataset.csv.


In [ ]:
candidate_models = get_classification_candidate_models(random_state=42)

Y_global = df_copy[target_col_class]
X_global = engineer_features(
    df_copy,
    Y_global,
    LOW_CATEGORICAL_FEATURES_OVERALL,
    HIGH_CATEGORICAL_FEATURES_GLOBAL
)

global_X_train, global_X_test, global_y_train, global_y_test = train_test_split(
    X_global, Y_global, test_size=0.2, random_state=42, stratify=Y_global
)

global_best_name_model, global_best_model, global_cv_scores = select_best_model_cv(
    global_X_train, global_y_train, candidate_models, cv_splits=cv_splits, scoring='f1_weighted'
)

global_predictions = global_best_model.predict(global_X_test)
global_feature_names = global_X_train.columns

In [ ]:
# Per-make models
# If the length of each dataframe of group is lower than the minimum sample variable, we shall pass and not create a 'per-make' model
# This is to make sure that algorithms like 'Random Forrest' gets trained on larger sets of data, as indented, if the condition is true, the make would fallback into the global model above.
for make, group_df in df.groupby("make"):
	if len(group_df) < min_samples_per_make:
		continue

	df_make_copy = df_copy.loc[group_df.index]

	# Prepare features for classification
	Y = df_make_copy[target_col_class]
	X = engineer_features(
		df_make_copy,
		Y,
		LOW_CATEGORICAL_FEATURES_OVERALL,
  		HIGH_CATEGORICAL_FEATURES_PER_MAKE
	)

	X_train, X_test, y_train, y_test = train_test_split(
		X, Y, test_size=0.2, random_state=42, stratify=Y
	)

	best_name, best_model, cv_scores = select_best_model_cv(
		X_train, y_train, candidate_models, cv_splits=cv_splits, scoring='f1_weighted'
	)
 
	predictions = best_model.predict(X_test)

	trained_models_per_make[make] = {
		"best_model_name": best_name,
		"model": best_model,
		"feature_names": X_train.columns,
		"X_test": X_test,
		"y_test": y_test,
		"predictions": predictions,
	}

In [ ]:
global_metrics = classification_metrics(global_y_test, global_predictions)

In [ ]:
print(f"\n Overall Dataset:")
print(f"  Best Model: {global_best_name_model}")
print(f"  Accuracy: {global_metrics['accuracy']:.3f}")
print(f"  Precision: {global_metrics['precision']:.3f}")
print(f"  Recall: {global_metrics['recall']:.3f}")
print(f"  F1-Score: {global_metrics['f1_score']:.3f}")

print(get_classification_report(
	global_y_test, 
	global_predictions, 
	target_names=['Budget', 'Mid-Range', 'Premium', 'Luxury']
))

for make_vehicle, data in trained_models_per_make.items():
	y_test = data['y_test']
	predictions = data['predictions']
	metrics = classification_metrics(y_test, predictions)

	print(f"\n{make_vehicle}:")
	print(f"  Best Model: {data['best_model_name']}")
	print(f"  Accuracy: {metrics['accuracy']:.3f}")
	print(f"  Precision: {metrics['precision']:.3f}")
	print(f"  Recall: {metrics['recall']:.3f}")
	print(f"  F1-Score: {metrics['f1_score']:.3f}")
 
	labels = y_test.cat.categories
	labels = [l for l in labels if l in y_test.unique()]
 
	print(get_classification_report(
		y_test, 
		predictions, 
		target_names=labels
	))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

def plot_feature_importances(values, feature_names, title):
    """Horizontal bar plot of feature importances/coefficients"""
    importance_df = (
        pd.DataFrame({
            "feature": feature_names,
            "importance": values
        })
        .sort_values("importance", ascending=False)
        .head(20)
    )

    plt.figure(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(importance_df)))
    plt.barh(importance_df["feature"], importance_df["importance"], 
             color=colors, edgecolor='black', linewidth=0.8)
    plt.gca().invert_yaxis()
    plt.xlabel('Importance', fontsize=12)
    plt.title(title, fontsize=16, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    

def plot_confusion_matrix(y_true, y_pred, labels=None, title="Confusion Matrix"):
    """Plot confusion matrix heatmap"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Count'})
    plt.title(title, fontsize=16, fontweight='bold')
    plt.ylabel('Actual', fontsize=12)
    plt.xlabel('Predicted', fontsize=12)
    plt.tight_layout()
    plt.show()

def plot_classification_metrics(metrics_dict, title="Classification Metrics"):
    """Bar plot of classification metrics"""
    # Extract metrics (excluding confusion matrix)
    plot_metrics = {k: v for k, v in metrics_dict.items() if k != 'confusion_matrix'}
    
    plt.figure(figsize=(10, 6))
    metrics_names = list(plot_metrics.keys())
    metrics_values = list(plot_metrics.values())
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    bars = plt.bar(metrics_names, metrics_values, color=colors[:len(metrics_names)])
    
    plt.ylim(0, 1.1)
    plt.ylabel('Score', fontsize=12)
    plt.title(title, fontsize=16, fontweight='bold')
    plt.xticks(fontsize=11)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_confusion_matrix(
	global_y_test, 
	global_predictions,
	labels=['Budget', 'Mid-Range', 'Premium', 'Luxury'],
	title=f"Confusion Matrix - Price Category Classification - Overall"
)

# Classification metrics bar chart
plot_classification_metrics(
	global_metrics,
	title=f"Classification Performance Metrics - Overall"
)

# Feature importances for classification
values_class, kind_class = get_feature_effects(global_best_model)
plot_feature_importances(
	values_class[:20],
	global_feature_names[:20],
	f"Feature {kind_class.title()}s - Classification - Overall"
)

In [ ]:
for make_vehicle, data in trained_models_per_make.items():
	model = data["model"]
	y_test = data['y_test']
	feature_names = data["feature_names"]
	predictions = data['predictions']
	metrics = classification_metrics(y_test, predictions)

	n = model.n_features_in_
	feature_names = feature_names[:n]

	plot_confusion_matrix(
		y_test, 
		predictions,
		labels=['Budget', 'Mid-Range', 'Premium', 'Luxury'],
		title=f"Confusion Matrix - Price Category Classification - {make_vehicle}"
	)

	# Classification metrics bar chart
	plot_classification_metrics(
		metrics,
		title=f"Classification Performance Metrics - {make_vehicle}"
	)

	# Feature importances for classification
	values_class, kind_class = get_feature_effects(model)
	plot_feature_importances(
		values_class[:20],
		feature_names[:20],
		f"Feature {kind_class.title()}s - Classification - {make_vehicle}"
	)